# Part 5 · Notebook 06 — Pivot levels, swings and the confirmation trap

**Sessions:** S9 (Levels & support/resistance) · S10 (Pivots & swings) · S11 (Chart patterns) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Compute floor pivot levels for today from yesterday's bar.
2. Find swing highs and lows (fractals) and when each becomes known.
3. See the biggest look-ahead bug in chart patterns: using the swing bar instead of the confirmation bar.
4. Prove a pattern detector is honest with a truncation test.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()

In [ ]:
df = p.synthetic_ohlcv(1500, seed=7)
o, h, l, c, v = p.arrays(df)

## 1. Floor pivots

Floor traders compute today's levels from **yesterday's** high, low and close: `P = (H + L + C)/3`, `R1 = 2P − L`, `S1 = 2P − H`, `R2 = P + (H − L)`, `S2 = P − (H − L)`. Shifting by one bar is what keeps them honest; bar 0 has no yesterday, so it is NaN.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def classic_pivots(high, low, close):
    h, l, c = (np.concatenate([[np.nan], a[:-1]]) for a in (high, low, close))   # yesterday's values
    pp = ...                                      # ✍️ the pivot
    return {"P": pp, "R1": ..., "S1": ...,        # ✍️ R1 and S1
            "R2": pp + (h - l), "S2": pp - (h - l)}

mine = p.attempt(classic_pivots, h, l, c)
mine = p.check("classic_pivots", mine, p.classic_pivots(h, l, c))
pd.DataFrame(mine).iloc[:4].round(3)

In [ ]:
lv = p.classic_pivots(h, l, c)
touch_r1 = np.nanmean(h[1:] >= lv["R1"][1:]); touch_s1 = np.nanmean(l[1:] <= lv["S1"][1:])
print(f"the day's high reaches R1 on {touch_r1:.0%} of days, the low reaches S1 on {touch_s1:.0%}")
print("on a random walk these are just distances of about one range; whether they act as levels is an empirical question")

## 2. Swing points (fractals) and when you know them

Bar `i` is a **swing high** if its high is *strictly* above the highs of the `k` bars on each side; a **swing low** likewise with lows. You can only know this after the `k` bars to the right have closed, so each swing is recorded with `known_from = i + k`. Append `p.Pivot(idx, known_from, price, kind)` with kind `-1` for a low and `+1` for a high (a low before a high at the same bar).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def fractals(high, low, k=2):
    out = []
    for i in range(k, len(high) - k):
        nb = np.r_[i - k:i, i + 1:i + k + 1]      # the k bars on each side
        if ...:                                   # ✍️ low[i] strictly below all the neighbours' lows
            out.append(p.Pivot(i, i + k, float(low[i]), -1))
        if ...:                                   # ✍️ high[i] strictly above all the neighbours' highs
            out.append(p.Pivot(i, i + k, float(high[i]), +1))
    return out

mine = fractals(h, l, 2)
mine = p.check("fractals", mine, p.fractals(h, l, 2))
print(f"{len(mine)} swings; the first three: {mine[:3]}")

## 3. The confirmation trap

A ZigZag swing needs price to reverse by `pct` before it is confirmed, which can take many bars. Draw the swings on a chart and they look like perfect turning points, because the chart shows them at the **swing bar**. A backtest that buys *there* is using the reversal that confirmed it.

In [ ]:
zz = p.zigzag(c, 0.05)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(c, lw=0.8, color="#8a8984")
for pv in zz[:14]:
    col = p.PALETTE[2] if pv.kind == -1 else p.PALETTE[7]
    ax.plot(pv.idx, pv.price, "o", color=col)
    ax.annotate("", xy=(pv.known_from, c[pv.known_from]), xytext=(pv.idx, pv.price), arrowprops={"arrowstyle": "->", "color": col, "lw": 0.8})
ax.set(xlim=(0, zz[13].known_from + 20), title="ZigZag 5%: the swing (dot) and the bar it becomes known (arrow tip)"); plt.show()
lag = np.array([pv.known_from - pv.idx for pv in zz])
print(f"{len(zz)} swings; confirmation arrives {np.median(lag):.0f} bars after the swing (median)")

In [ ]:
rows = []
for name, piv in [("ZigZag 5%", zz), ("fractal k=2", p.fractals(h, l, 2)), ("fractal k=5", p.fractals(h, l, 5))]:
    for when, flag in [("swing bar (bug)", False), ("confirmation bar", True)]:
        r = p.swing_low_trades(piv, o, horizon=10, use_known_from=flag)
        rows.append({"swings": name, "buy after": when, "trades": len(r), "mean 10-bar return": f"{r.mean():+.2%}"})
pd.DataFrame(rows)

Buying the swing low "at" the swing bar looks like an edge in every detector, largest for ZigZag. Waiting for confirmation, as a live system must, the edge is gone.

## 4. The truncation test

The general cure: run the detector on the data **up to bar t** and check that every swing it reports as known by `t` is also in the full-data result. A detector that uses the future will report different swings when the future is cut off.

In [ ]:
def truncation_ok(detector, x, cuts):
    full = detector(x)
    for t in cuts:
        known_full = [pv for pv in full if pv.known_from <= t]
        known_cut = [pv for pv in detector(x[: t + 1]) if pv.known_from <= t]
        if known_full != known_cut:
            return False
    return True

def zigzag_with_last_swing(x, pct=0.05):
    """A tempting variant: also report the latest extreme as a swing, 'known' at its own bar."""
    out = p.zigzag(x, pct)
    tail = x[out[-1].known_from:] if out else x
    j = int(np.argmin(tail)) + (out[-1].known_from if out else 0)
    return out + [p.Pivot(j, j, float(x[j]), -1)]

cuts = range(200, 1500, 50)
print("zigzag:                ", truncation_ok(lambda x: p.zigzag(x, 0.05), c, cuts))
print("fractals k=2:          ", truncation_ok(lambda x: p.fractals(x, x, 2), c, cuts))
print("zigzag + 'latest low': ", truncation_ok(zigzag_with_last_swing, c, cuts))

## Wrap-up

* Levels from yesterday's bar are known today; swings are known only at confirmation.
* Store `known_from` with every swing and trade from it, never from the swing bar.
* Truncation tests catch the leak whatever the detector does inside.
* Graded versions: `labs/part05/week19_patterns` (pivots, ZigZag, fractals, double tops, head & shoulders) and the Clinic W3 scanner audit.